# Healthcare Readmission Analytics
## 02 — Data Preprocessing

### Objective

Transform the audited raw hospital encounter data into a reproducible, analysis-ready dataset while preserving the original source files.

This notebook implements the preprocessing decisions established in `01_data_audit.ipynb`, including:

- standardizing source-level missing-value representations;
- preserving clinically and administratively meaningful unknown categories;
- decoding administrative lookup fields;
- constructing the binary 30-day readmission target;
- separating identifiers from predictive features;
- identifying unusable or extremely sparse features;
- preserving patient identifiers for patient-aware train/test partitioning;
- validating data integrity after transformation; and
- exporting reproducible interim and processed datasets for PostgreSQL, statistical analysis, and predictive modeling.

> **Data integrity principle:** Files under `data/raw/` are immutable. All transformations are performed on copies and written to downstream directories.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the project root whether the notebook starts from
# the repository root or the notebooks/ directory.
cwd = Path.cwd().resolve()

if (cwd / "data" / "raw").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate the project root containing data/raw.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Ensure downstream output directories exist.
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data:     {RAW_DIR}")
print(f"Interim data: {INTERIM_DIR}")
print(f"Processed:    {PROCESSED_DIR}")

Project root: C:\Users\amohe.000\Downloads\healthcare-readmission-analytics
Raw data:     C:\Users\amohe.000\Downloads\healthcare-readmission-analytics\data\raw
Interim data: C:\Users\amohe.000\Downloads\healthcare-readmission-analytics\data\interim
Processed:    C:\Users\amohe.000\Downloads\healthcare-readmission-analytics\data\processed


In [2]:
# Inventory raw source files before preprocessing

raw_files = sorted(path for path in RAW_DIR.iterdir() if path.is_file())

print(f"Raw files found: {len(raw_files)}")
print("-" * 60)

for path in raw_files:
    size_mb = path.stat().st_size / (1024**2)
    print(f"{path.name:<35} {size_mb:>8.2f} MB")

Raw files found: 2
------------------------------------------------------------
diabetic_data.csv                      18.27 MB
IDS_mapping.csv                         0.00 MB


In [3]:
# Load the raw encounter dataset without altering source representations

DATA_FILE = RAW_DIR / "diabetic_data.csv"
MAPPING_FILE = RAW_DIR / "IDS_mapping.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Encounter dataset not found: {DATA_FILE}")

if not MAPPING_FILE.exists():
    raise FileNotFoundError(f"Mapping file not found: {MAPPING_FILE}")

df_raw = pd.read_csv(DATA_FILE, keep_default_na=True)

print("Raw encounter dataset loaded successfully.")
print(f"Rows:    {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")
print(f"Memory:  {df_raw.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB")

Raw encounter dataset loaded successfully.
Rows:    101,766
Columns: 50
Memory:  192.87 MB


In [4]:
# Verify source-level question-mark missingness before preprocessing

question_mark_counts = df_raw.astype(str).eq("?").sum()

question_mark_counts = (
    question_mark_counts[question_mark_counts > 0]
    .sort_values(ascending=False)
    .rename("question_mark_count")
    .to_frame()
)

question_mark_counts["question_mark_pct"] = (
    question_mark_counts["question_mark_count"] / len(df_raw) * 100
).round(2)

print("Columns containing literal '?' markers:")
print(f"Columns affected: {len(question_mark_counts)}")
print(f"Total '?' cells: " f"{question_mark_counts['question_mark_count'].sum():,}")

question_mark_counts

Columns containing literal '?' markers:
Columns affected: 7
Total '?' cells: 192,849


,question_mark_count,question_mark_pct
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [5]:
# Create a preprocessing working copy and standardize '?' as missing values

df_work = df_raw.copy(deep=True)

question_mark_columns = question_mark_counts.index.tolist()

df_work[question_mark_columns] = df_work[question_mark_columns].replace("?", pd.NA)

remaining_question_marks = (
    df_work[question_mark_columns].astype(str).eq("?").sum().sum()
)

standardized_missing_counts = (
    df_work[question_mark_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

standardized_missing_counts["missing_pct"] = (
    standardized_missing_counts["missing_count"] / len(df_work) * 100
).round(2)

print("Working preprocessing copy created.")
print(f"Shape: {df_work.shape}")
print(f"Columns standardized: {len(question_mark_columns)}")
print(f"Remaining literal '?' cells: {remaining_question_marks:,}")

standardized_missing_counts

Working preprocessing copy created.
Shape: (101766, 50)
Columns standardized: 7
Remaining literal '?' cells: 0


,missing_count,missing_pct
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [6]:
# Validate that preprocessing did not mutate the raw in-memory dataset

raw_question_marks_after_copy = df_raw.astype(str).eq("?").sum()

raw_question_marks_after_copy = raw_question_marks_after_copy[
    raw_question_marks_after_copy > 0
].sort_values(ascending=False)

raw_total_question_marks = int(raw_question_marks_after_copy.sum())

shape_preserved = df_raw.shape == df_work.shape
raw_markers_preserved = raw_total_question_marks == int(
    question_mark_counts["question_mark_count"].sum()
)
working_markers_removed = remaining_question_marks == 0

print("Raw-data immutability validation")
print("-" * 40)
print(f"Raw shape:                 {df_raw.shape}")
print(f"Working shape:             {df_work.shape}")
print(f"Shape preserved:           {shape_preserved}")
print(f"Raw '?' cells preserved:   {raw_total_question_marks:,}")
print(f"Raw markers unchanged:     {raw_markers_preserved}")
print(f"Working '?' cells:         {remaining_question_marks:,}")
print(f"Working markers removed:   {working_markers_removed}")

assert shape_preserved, "Row/column shape changed unexpectedly."
assert raw_markers_preserved, "df_raw appears to have been modified."
assert working_markers_removed, "Literal '?' markers remain in df_work."

print("\nValidation passed: df_raw remains unchanged.")

Raw-data immutability validation
----------------------------------------
Raw shape:                 (101766, 50)
Working shape:             (101766, 50)
Shape preserved:           True
Raw '?' cells preserved:   192,849
Raw markers unchanged:     True
Working '?' cells:         0
Working markers removed:   True

Validation passed: df_raw remains unchanged.


In [7]:
# Audit total missingness after standardizing source markers

missing_after_standardization = df_work.isna().sum().rename("missing_count").to_frame()

missing_after_standardization = missing_after_standardization[
    missing_after_standardization["missing_count"] > 0
].copy()

missing_after_standardization["missing_pct"] = (
    missing_after_standardization["missing_count"] / len(df_work) * 100
).round(2)

missing_after_standardization = missing_after_standardization.sort_values(
    "missing_count", ascending=False
)

total_missing_cells = int(missing_after_standardization["missing_count"].sum())

print("Missingness after source-marker standardization")
print("-" * 50)
print(f"Columns with missing values: {len(missing_after_standardization)}")
print(f"Total missing cells:         {total_missing_cells:,}")

missing_after_standardization

Missingness after source-marker standardization
--------------------------------------------------
Columns with missing values: 9
Total missing cells:         374,017


,missing_count,missing_pct
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [8]:
# Define missing-data treatment decisions from the completed audit

missing_treatment_plan = pd.DataFrame(
    [
        {
            "column": "weight",
            "missing_pct": 96.86,
            "treatment": "drop_feature",
            "rationale": (
                "Nearly all observations are missing; insufficient "
                "information remains for reliable modeling."
            ),
        },
        {
            "column": "max_glu_serum",
            "missing_pct": 94.75,
            "treatment": "preserve_missing_category",
            "rationale": (
                "Missingness primarily indicates that the laboratory "
                "test was not recorded/performed rather than an unknown "
                "continuous measurement."
            ),
        },
        {
            "column": "A1Cresult",
            "missing_pct": 83.28,
            "treatment": "preserve_missing_category",
            "rationale": (
                "Missingness carries clinical/process information and "
                "should remain distinguishable from observed test results."
            ),
        },
        {
            "column": "medical_specialty",
            "missing_pct": 49.08,
            "treatment": "explicit_unknown",
            "rationale": (
                "Retain the feature while representing unavailable "
                "specialty information explicitly."
            ),
        },
        {
            "column": "payer_code",
            "missing_pct": 39.56,
            "treatment": "explicit_unknown",
            "rationale": (
                "Retain unavailable payer information as an explicit "
                "category rather than statistically imputing it."
            ),
        },
        {
            "column": "race",
            "missing_pct": 2.23,
            "treatment": "explicit_unknown",
            "rationale": (
                "Preserve encounters and represent unavailable race "
                "information explicitly rather than deleting patients."
            ),
        },
        {
            "column": "diag_3",
            "missing_pct": 1.40,
            "treatment": "explicit_missing",
            "rationale": (
                "Absence of a tertiary diagnosis is meaningful and "
                "should not be statistically fabricated."
            ),
        },
        {
            "column": "diag_2",
            "missing_pct": 0.35,
            "treatment": "explicit_missing",
            "rationale": ("Absence of a secondary diagnosis is retained explicitly."),
        },
        {
            "column": "diag_1",
            "missing_pct": 0.02,
            "treatment": "explicit_missing",
            "rationale": (
                "Rare missing primary diagnoses are retained explicitly "
                "without inventing a diagnosis."
            ),
        },
    ]
)

# Verify that the treatment register covers every field with missingness.
audited_missing_columns = set(missing_after_standardization.index)
planned_missing_columns = set(missing_treatment_plan["column"])

assert audited_missing_columns == planned_missing_columns, (
    "Missing-data treatment register does not exactly match "
    "the audited missing columns."
)

print("Missing-data treatment register")
print("-" * 40)
print(f"Audited missing columns: {len(audited_missing_columns)}")
print(f"Treatment decisions:     {len(planned_missing_columns)}")
print("Coverage complete:       True")

missing_treatment_plan

Missing-data treatment register
----------------------------------------
Audited missing columns: 9
Treatment decisions:     9
Coverage complete:       True


,column,missing_pct,treatment,rationale
0,weight,96.86,drop_feature,Nearly all observations are missing; insuffici...
1,max_glu_serum,94.75,preserve_missing_category,Missingness primarily indicates that the labor...
2,A1Cresult,83.28,preserve_missing_category,Missingness carries clinical/process informati...
3,medical_specialty,49.08,explicit_unknown,Retain the feature while representing unavaila...
4,payer_code,39.56,explicit_unknown,Retain unavailable payer information as an exp...
5,race,2.23,explicit_unknown,Preserve encounters and represent unavailable ...
6,diag_3,1.40,explicit_missing,Absence of a tertiary diagnosis is meaningful ...
7,diag_2,0.35,explicit_missing,Absence of a secondary diagnosis is retained e...
8,diag_1,0.02,explicit_missing,Rare missing primary diagnoses are retained ex...


In [9]:
# Apply documented missing-data treatments to the working dataset

# 1. Drop the effectively unusable weight feature.
df_work = df_work.drop(columns=["weight"])

# 2. Preserve missing laboratory-test states as explicit categories.
for column in ["max_glu_serum", "A1Cresult"]:
    df_work[column] = df_work[column].fillna("Not recorded")

# 3. Represent unavailable administrative/demographic categories explicitly.
for column in ["medical_specialty", "payer_code", "race"]:
    df_work[column] = df_work[column].fillna("Unknown")

# 4. Preserve absent diagnosis information without fabricating diagnoses.
for column in ["diag_1", "diag_2", "diag_3"]:
    df_work[column] = df_work[column].fillna("Missing")

# Validate the transformation.
remaining_missing = df_work.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("Missing-data treatment applied")
print("-" * 40)
print(f"Working rows:              {df_work.shape[0]:,}")
print(f"Working columns:           {df_work.shape[1]}")
print(f"'weight' removed:          {'weight' not in df_work.columns}")
print(f"Columns still containing NaN: {len(remaining_missing)}")
print(f"Total remaining NaN cells: {remaining_missing.sum():,}")

remaining_missing

Missing-data treatment applied
----------------------------------------
Working rows:              101,766
Working columns:           49
'weight' removed:          True
Columns still containing NaN: 0
Total remaining NaN cells: 0


Series([], dtype: int64)

In [10]:
# Verify explicit missing-value categories created during preprocessing

replacement_categories = {
    "max_glu_serum": "Not recorded",
    "A1Cresult": "Not recorded",
    "medical_specialty": "Unknown",
    "payer_code": "Unknown",
    "race": "Unknown",
    "diag_1": "Missing",
    "diag_2": "Missing",
    "diag_3": "Missing",
}

replacement_validation = []

for column, replacement in replacement_categories.items():
    count = df_work[column].eq(replacement).sum()
    pct = count / len(df_work) * 100

    replacement_validation.append(
        {
            "column": column,
            "replacement_category": replacement,
            "count": int(count),
            "pct": round(pct, 2),
        }
    )

replacement_validation = pd.DataFrame(replacement_validation)

print("Explicit missing-category validation")
print("-" * 45)
print(f"Columns validated: {len(replacement_validation)}")
print(
    "Total explicit replacement values: " f"{replacement_validation['count'].sum():,}"
)

replacement_validation

Explicit missing-category validation
---------------------------------------------
Columns validated: 8
Total explicit replacement values: 275,448


,column,replacement_category,count,pct
0,max_glu_serum,Not recorded,96420,94.75
1,A1Cresult,Not recorded,84748,83.28
2,medical_specialty,Unknown,49949,49.08
3,payer_code,Unknown,40256,39.56
4,race,Unknown,2273,2.23
5,diag_1,Missing,21,0.02
6,diag_2,Missing,358,0.35
7,diag_3,Missing,1423,1.40


In [11]:
# Reconcile audited missingness with preprocessing treatment

original_missing_cells = int(missing_after_standardization["missing_count"].sum())
dropped_weight_missing = int(
    missing_after_standardization.loc["weight", "missing_count"]
)
explicit_replacements = int(replacement_validation["count"].sum())

reconciled_missing_cells = dropped_weight_missing + explicit_replacements
reconciliation_passed = reconciled_missing_cells == original_missing_cells

print("Missing-data reconciliation")
print("-" * 40)
print(f"Original missing cells:        {original_missing_cells:,}")
print(f"Dropped with 'weight':         {dropped_weight_missing:,}")
print(f"Explicit category replacements:{explicit_replacements:>10,}")
print(f"Accounted-for missing cells:   {reconciled_missing_cells:,}")
print(f"Reconciliation passed:         {reconciliation_passed}")

assert reconciliation_passed, (
    "Missing-data preprocessing does not reconcile with the audited "
    "missing-cell total."
)

Missing-data reconciliation
----------------------------------------
Original missing cells:        374,017
Dropped with 'weight':         98,569
Explicit category replacements:   275,448
Accounted-for missing cells:   374,017
Reconciliation passed:         True


In [12]:
# Load administrative lookup mapping for preprocessing

ids_mapping = pd.read_csv(
    MAPPING_FILE,
    header=None,
    names=["raw_text"],
    dtype=str,
    keep_default_na=False,
)

print("Administrative mapping source loaded")
print("-" * 45)
print(f"Rows: {len(ids_mapping):,}")
print(f"Columns: {ids_mapping.shape[1]}")
print(f"Source: {MAPPING_FILE.name}")

ids_mapping.head(15)

Administrative mapping source loaded
---------------------------------------------
Rows: 68
Columns: 1
Source: IDS_mapping.csv


,raw_text
admission_type_id,description
1,Emergency
2,Urgent
3,Elective
4,Newborn
5,Not Available
6,NULL
7,Trauma Center
8,Not Mapped
,


In [13]:
# Parse administrative lookup sections from the raw mapping file

mapping_lines = MAPPING_FILE.read_text(encoding="utf-8-sig").splitlines()

section_headers = {
    "admission_type_id": "admission_type_id",
    "discharge_disposition_id": "discharge_disposition_id",
    "admission_source_id": "admission_source_id",
}

lookup_records = {
    "admission_type_id": [],
    "discharge_disposition_id": [],
    "admission_source_id": [],
}

current_section = None

for line in mapping_lines:
    line = line.strip()

    if not line:
        continue

    first_field = line.split(",", 1)[0].strip()

    if first_field in section_headers:
        current_section = section_headers[first_field]
        continue

    # Skip each section's column-header row.
    if first_field.lower() in {
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
    }:
        continue

    if current_section is None:
        continue

    parts = line.split(",", 1)

    if len(parts) != 2:
        continue

    code_text, description = parts
    code_text = code_text.strip()
    description = description.strip()

    if code_text.isdigit():
        lookup_records[current_section].append(
            {
                "code": int(code_text),
                "description": description,
            }
        )

lookup_tables = {
    name: pd.DataFrame(records) for name, records in lookup_records.items()
}

print("Administrative lookup sections parsed")
print("-" * 45)

for name, table in lookup_tables.items():
    print(
        f"{name:<28} "
        f"{len(table):>2} codes | "
        f"unique codes: {table['code'].nunique():>2}"
    )

Administrative lookup sections parsed
---------------------------------------------
admission_type_id             8 codes | unique codes:  8
discharge_disposition_id     30 codes | unique codes: 30
admission_source_id          25 codes | unique codes: 25


In [14]:
# Validate administrative lookup coverage against encounter data

lookup_coverage = []

for column, table in lookup_tables.items():
    encounter_codes = set(
        pd.to_numeric(df_work[column], errors="coerce").dropna().astype(int).unique()
    )

    mapped_codes = set(table["code"].astype(int))

    unmapped_codes = sorted(encounter_codes - mapped_codes)
    unused_mapping_codes = sorted(mapped_codes - encounter_codes)

    lookup_coverage.append(
        {
            "column": column,
            "encounter_unique_codes": len(encounter_codes),
            "mapping_codes": len(mapped_codes),
            "unmapped_encounter_codes": unmapped_codes,
            "unused_mapping_codes": unused_mapping_codes,
            "coverage_complete": len(unmapped_codes) == 0,
        }
    )

lookup_coverage = pd.DataFrame(lookup_coverage)

print("Administrative lookup coverage validation")
print("-" * 50)

for row in lookup_coverage.itertuples(index=False):
    print(f"{row.column}")
    print(f"  Encounter codes:       {row.encounter_unique_codes}")
    print(f"  Mapping codes:         {row.mapping_codes}")
    print(f"  Unmapped encounter:    {row.unmapped_encounter_codes}")
    print(f"  Unused mapping codes:  {row.unused_mapping_codes}")
    print(f"  Coverage complete:     {row.coverage_complete}")
    print()

assert lookup_coverage["coverage_complete"].all(), (
    "At least one administrative code appearing in the encounter "
    "dataset is absent from its reference lookup."
)

lookup_coverage

Administrative lookup coverage validation
--------------------------------------------------
admission_type_id
  Encounter codes:       8
  Mapping codes:         8
  Unmapped encounter:    []
  Unused mapping codes:  []
  Coverage complete:     True

discharge_disposition_id
  Encounter codes:       26
  Mapping codes:         30
  Unmapped encounter:    []
  Unused mapping codes:  [21, 26, 29, 30]
  Coverage complete:     True

admission_source_id
  Encounter codes:       17
  Mapping codes:         25
  Unmapped encounter:    []
  Unused mapping codes:  [12, 15, 18, 19, 21, 23, 24, 26]
  Coverage complete:     True



,column,encounter_unique_codes,mapping_codes,unmapped_encounter_codes,unused_mapping_codes,coverage_complete
0,admission_type_id,8,8,[],[],True
1,discharge_disposition_id,26,30,[],"[21, 26, 29, 30]",True
2,admission_source_id,17,25,[],"[12, 15, 18, 19, 21, 23, 24, 26]",True


In [15]:
# Decode administrative lookup fields while preserving original ID columns

decoded_columns = {}

for column, table in lookup_tables.items():
    description_column = column.replace("_id", "_description")

    mapping_dict = dict(
        zip(table["code"].astype(int), table["description"].astype(str))
    )

    numeric_codes = pd.to_numeric(df_work[column], errors="raise").astype(int)

    decoded = numeric_codes.map(mapping_dict)

    # Source metadata uses literal "NULL" as the description
    # for several valid administrative codes.
    decoded = decoded.replace("NULL", "Unknown")

    df_work[description_column] = decoded

    decoded_columns[column] = description_column


print("Administrative fields decoded")
print("-" * 45)

for source_column, description_column in decoded_columns.items():
    print(f"{source_column:<28} -> " f"{description_column}")

print()
print(f"Working rows:    {len(df_work):,}")
print(f"Working columns: {df_work.shape[1]}")

Administrative fields decoded
---------------------------------------------
admission_type_id            -> admission_type_description
discharge_disposition_id     -> discharge_disposition_description
admission_source_id          -> admission_source_description

Working rows:    101,766
Working columns: 52


In [16]:
# Validate decoded administrative descriptions

admin_validation = []

for source_column, description_column in decoded_columns.items():
    missing_descriptions = int(df_work[description_column].isna().sum())
    unknown_count = int((df_work[description_column] == "Unknown").sum())
    unique_ids = int(df_work[source_column].nunique())
    unique_descriptions = int(df_work[description_column].nunique())

    admin_validation.append(
        {
            "id_column": source_column,
            "description_column": description_column,
            "unique_ids": unique_ids,
            "unique_descriptions": unique_descriptions,
            "missing_descriptions": missing_descriptions,
            "unknown_encounters": unknown_count,
        }
    )

admin_validation = pd.DataFrame(admin_validation)

print("Administrative decoding validation")
print("-" * 45)
print(f"Rows preserved: {len(df_work):,}")
print("Unmapped descriptions: " f"{admin_validation['missing_descriptions'].sum():,}")
print(
    "Source-defined unknown encounters: "
    f"{admin_validation['unknown_encounters'].sum():,}"
)

assert (
    admin_validation["missing_descriptions"].sum() == 0
), "Administrative decoding introduced unmapped descriptions."

admin_validation

Administrative decoding validation
---------------------------------------------
Rows preserved: 101,766
Unmapped descriptions: 0
Source-defined unknown encounters: 15,763


,id_column,description_column,unique_ids,unique_descriptions,missing_descriptions,unknown_encounters
0,admission_type_id,admission_type_description,8,8,0,5291
1,discharge_disposition_id,discharge_disposition_description,26,26,0,3691
2,admission_source_id,admission_source_description,17,17,0,6781


In [17]:
# Construct binary 30-day readmission target while preserving source outcome

print("Source readmission outcome")
print("-" * 40)

source_readmission_counts = (
    df_work["readmitted"]
    .value_counts(dropna=False)
    .rename_axis("readmitted")
    .to_frame("count")
)

source_readmission_counts["pct"] = (
    source_readmission_counts["count"] / len(df_work) * 100
).round(2)

source_readmission_counts

Source readmission outcome
----------------------------------------


,count,pct
readmitted,,
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16


In [18]:
# Create binary target for readmission within 30 days

df_work["readmitted_30d"] = df_work["readmitted"].eq("<30").astype("int8")

target_distribution = (
    df_work["readmitted_30d"]
    .value_counts()
    .sort_index()
    .rename_axis("readmitted_30d")
    .to_frame("count")
)

target_distribution["pct"] = (target_distribution["count"] / len(df_work) * 100).round(
    2
)

positive_cases = int(df_work["readmitted_30d"].sum())
expected_positive_cases = int((df_work["readmitted"] == "<30").sum())

print("Binary 30-day readmission target")
print("-" * 40)
print(f"Rows:                    {len(df_work):,}")
print(f"Positive cases:          {positive_cases:,}")
print(f"Expected positive cases: {expected_positive_cases:,}")
print(f"Target matches source:   {positive_cases == expected_positive_cases}")
print(f"Target missing values:   {df_work['readmitted_30d'].isna().sum():,}")
print(f"Working columns:         {df_work.shape[1]}")

assert (
    positive_cases == expected_positive_cases
), "Binary target does not reconcile with source '<30' outcome."

assert (
    df_work["readmitted_30d"].isna().sum() == 0
), "Binary target contains missing values."

assert set(df_work["readmitted_30d"].unique()) <= {
    0,
    1,
}, "Binary target contains values other than 0 and 1."

target_distribution

Binary 30-day readmission target
----------------------------------------
Rows:                    101,766
Positive cases:          11,357
Expected positive cases: 11,357
Target matches source:   True
Target missing values:   0
Working columns:         53


,count,pct
readmitted_30d,,
0,90409,88.84
1,11357,11.16


In [19]:
# Validate binary target against source outcome at row level

expected_target = df_work["readmitted"].eq("<30").astype("int8")

target_mismatches = df_work.loc[
    df_work["readmitted_30d"] != expected_target,
    ["encounter_id", "patient_nbr", "readmitted", "readmitted_30d"],
]

mapping_check = pd.crosstab(
    df_work["readmitted"], df_work["readmitted_30d"], margins=True
)

print("30-day target row-level validation")
print("-" * 45)
print(f"Rows validated:       {len(df_work):,}")
print(f"Target mismatches:    {len(target_mismatches):,}")
print(f"Row-level validation: {len(target_mismatches) == 0}")

assert (
    len(target_mismatches) == 0
), "Binary target does not match the source outcome for every row."

mapping_check

30-day target row-level validation
---------------------------------------------
Rows validated:       101,766
Target mismatches:    0
Row-level validation: True


readmitted_30d,0,1,All
readmitted,,,
<30,0,11357,11357
>30,35545,0,35545
NO,54864,0,54864
All,90409,11357,101766


In [20]:
# Define identifier, target, and source-outcome roles

identifier_columns = [
    "encounter_id",
    "patient_nbr",
]

target_column = "readmitted_30d"

source_outcome_columns = [
    "readmitted",
]

protected_columns = identifier_columns + source_outcome_columns + [target_column]

missing_protected_columns = [
    column for column in protected_columns if column not in df_work.columns
]

candidate_feature_columns = [
    column for column in df_work.columns if column not in protected_columns
]

print("Modeling column-role definition")
print("-" * 45)
print(f"Working columns:            {df_work.shape[1]}")
print(f"Identifier columns:         {len(identifier_columns)}")
print(f"Source outcome columns:     {len(source_outcome_columns)}")
print(f"Target columns:             1")
print(f"Candidate feature columns:  {len(candidate_feature_columns)}")
print(f"Missing protected columns:  {missing_protected_columns}")

print("\nIdentifiers:")
for column in identifier_columns:
    print(f"  - {column}")

print(f"\nSource outcome: {source_outcome_columns[0]}")
print(f"Target:         {target_column}")

assert (
    not missing_protected_columns
), "One or more required identifier/outcome columns are missing."

assert not set(identifier_columns) & set(
    candidate_feature_columns
), "Identifier leakage detected in candidate features."

assert (
    target_column not in candidate_feature_columns
), "Target leakage detected in candidate features."

assert not set(source_outcome_columns) & set(
    candidate_feature_columns
), "Source outcome leakage detected in candidate features."

Modeling column-role definition
---------------------------------------------
Working columns:            53
Identifier columns:         2
Source outcome columns:     1
Target columns:             1
Candidate feature columns:  49
Missing protected columns:  []

Identifiers:
  - encounter_id
  - patient_nbr

Source outcome: readmitted
Target:         readmitted_30d


In [21]:
# Audit candidate features for cardinality and constant columns

feature_audit = pd.DataFrame(
    {
        "column": candidate_feature_columns,
        "dtype": [str(df_work[column].dtype) for column in candidate_feature_columns],
        "unique_values": [
            df_work[column].nunique(dropna=False)
            for column in candidate_feature_columns
        ],
        "missing_values": [
            int(df_work[column].isna().sum()) for column in candidate_feature_columns
        ],
    }
)

feature_audit["missing_pct"] = (
    feature_audit["missing_values"] / len(df_work) * 100
).round(2)

feature_audit["constant"] = feature_audit["unique_values"] <= 1

constant_features = feature_audit.loc[feature_audit["constant"], "column"].tolist()

print("Candidate-feature audit")
print("-" * 45)
print(f"Candidate features:       {len(candidate_feature_columns)}")
print(f"Constant features:        {len(constant_features)}")
print(f"Features containing NaN:  {(feature_audit['missing_values'] > 0).sum()}")
print(f"Constant columns:         {constant_features}")

assert set(constant_features) == {
    "examide",
    "citoglipton",
}, "Constant-feature set differs from the completed raw-data audit."

feature_audit.sort_values(
    ["constant", "unique_values"], ascending=[False, True]
).reset_index(drop=True)

Candidate-feature audit
---------------------------------------------
Candidate features:       49
Constant features:        2
Features containing NaN:  0
Constant columns:         ['examide', 'citoglipton']


,column,dtype,unique_values,missing_values,missing_pct,constant
0,examide,str,1,0,0.0,True
1,citoglipton,str,1,0,0.0,True
2,acetohexamide,str,2,0,0.0,False
3,tolbutamide,str,2,0,0.0,False
4,troglitazone,str,2,0,0.0,False
5,glipizide-metformin,str,2,0,0.0,False
6,glimepiride-pioglitazone,str,2,0,0.0,False
7,metformin-rosiglitazone,str,2,0,0.0,False
8,metformin-pioglitazone,str,2,0,0.0,False
9,change,str,2,0,0.0,False


In [22]:
# Remove audited zero-variance features from modeling candidates

excluded_constant_features = constant_features.copy()

model_feature_columns = [
    column
    for column in candidate_feature_columns
    if column not in excluded_constant_features
]

print("Model-feature eligibility after zero-variance removal")
print("-" * 55)
print(f"Candidate features before: {len(candidate_feature_columns)}")
print(f"Constant features removed: {len(excluded_constant_features)}")
print(f"Eligible model features:    {len(model_feature_columns)}")
print(f"Removed features:           {excluded_constant_features}")

assert len(model_feature_columns) == 47, "Unexpected number of model-eligible features."

assert not set(excluded_constant_features) & set(
    model_feature_columns
), "A zero-variance feature remains in the model feature set."

assert not set(identifier_columns) & set(
    model_feature_columns
), "Identifier leakage detected in model features."

assert (
    target_column not in model_feature_columns
), "Target leakage detected in model features."

assert not set(source_outcome_columns) & set(
    model_feature_columns
), "Source-outcome leakage detected in model features."

assert (
    df_work[model_feature_columns].isna().sum().sum() == 0
), "Missing values remain in model-eligible features."

print("\nValidation passed: model-feature eligibility is clean.")

Model-feature eligibility after zero-variance removal
-------------------------------------------------------
Candidate features before: 49
Constant features removed: 2
Eligible model features:    47
Removed features:           ['examide', 'citoglipton']

Validation passed: model-feature eligibility is clean.


In [23]:
# Classify model-eligible features by semantic role

administrative_id_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

administrative_description_features = [
    "admission_type_description",
    "discharge_disposition_description",
    "admission_source_description",
]

quantitative_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

categorical_features = [
    column for column in model_feature_columns if column not in quantitative_features
]

role_registry = pd.DataFrame(
    [
        {
            "column": column,
            "role": (
                "quantitative" if column in quantitative_features else "categorical"
            ),
            "dtype": str(df_work[column].dtype),
            "unique_values": int(df_work[column].nunique(dropna=False)),
        }
        for column in model_feature_columns
    ]
)

print("Model-feature semantic-role classification")
print("-" * 55)
print(f"Eligible model features: {len(model_feature_columns)}")
print(f"Quantitative features:   {len(quantitative_features)}")
print(f"Categorical features:    {len(categorical_features)}")
print(
    "Role coverage complete: "
    f"{len(quantitative_features) + len(categorical_features) == len(model_feature_columns)}"
)

assert set(quantitative_features).issubset(
    model_feature_columns
), "A quantitative feature is missing from the eligible feature set."

assert set(categorical_features).isdisjoint(
    quantitative_features
), "A feature was assigned to both categorical and quantitative roles."

assert set(categorical_features) | set(quantitative_features) == set(
    model_feature_columns
), "Feature-role classification does not cover every model feature."

role_registry.sort_values(["role", "unique_values", "column"]).reset_index(drop=True)

Model-feature semantic-role classification
-------------------------------------------------------
Eligible model features: 47
Quantitative features:   8
Categorical features:    39
Role coverage complete: True


,column,role,dtype,unique_values
0,acetohexamide,categorical,str,2
1,change,categorical,str,2
2,diabetesMed,categorical,str,2
3,glimepiride-pioglitazone,categorical,str,2
4,glipizide-metformin,categorical,str,2
5,metformin-pioglitazone,categorical,str,2
6,metformin-rosiglitazone,categorical,str,2
7,tolbutamide,categorical,str,2
8,troglitazone,categorical,str,2
9,gender,categorical,str,3


In [24]:
# Audit redundancy between administrative IDs and decoded descriptions

administrative_pairs = {
    "admission_type_id": "admission_type_description",
    "discharge_disposition_id": "discharge_disposition_description",
    "admission_source_id": "admission_source_description",
}

redundancy_records = []

for id_column, description_column in administrative_pairs.items():
    id_to_description = (
        df_work[[id_column, description_column]]
        .drop_duplicates()
        .groupby(id_column, dropna=False)[description_column]
        .nunique(dropna=False)
    )

    description_to_id = (
        df_work[[id_column, description_column]]
        .drop_duplicates()
        .groupby(description_column, dropna=False)[id_column]
        .nunique(dropna=False)
    )

    max_descriptions_per_id = int(id_to_description.max())
    max_ids_per_description = int(description_to_id.max())

    redundancy_records.append(
        {
            "id_column": id_column,
            "description_column": description_column,
            "unique_ids": int(df_work[id_column].nunique(dropna=False)),
            "unique_descriptions": int(
                df_work[description_column].nunique(dropna=False)
            ),
            "max_descriptions_per_id": max_descriptions_per_id,
            "max_ids_per_description": max_ids_per_description,
            "id_determines_description": max_descriptions_per_id == 1,
        }
    )

administrative_redundancy = pd.DataFrame(redundancy_records)

print("Administrative ID/description redundancy audit")
print("-" * 55)
print(f"Administrative pairs audited: {len(administrative_pairs)}")
print(
    "Every ID deterministically maps to one description: "
    f"{administrative_redundancy['id_determines_description'].all()}"
)

assert administrative_redundancy[
    "id_determines_description"
].all(), "At least one administrative ID maps to multiple descriptions."

administrative_redundancy

Administrative ID/description redundancy audit
-------------------------------------------------------
Administrative pairs audited: 3
Every ID deterministically maps to one description: True


,id_column,description_column,unique_ids,unique_descriptions,max_descriptions_per_id,max_ids_per_description,id_determines_description
0,admission_type_id,admission_type_description,8,8,1,1,True
1,discharge_disposition_id,discharge_disposition_description,26,26,1,1,True
2,admission_source_id,admission_source_description,17,17,1,1,True


In [25]:
# Finalize model feature registry by removing redundant administrative IDs

redundant_administrative_ids = list(administrative_pairs.keys())

final_model_features = [
    column
    for column in model_feature_columns
    if column not in redundant_administrative_ids
]

final_quantitative_features = [
    column for column in quantitative_features if column in final_model_features
]

final_categorical_features = [
    column for column in categorical_features if column in final_model_features
]

print("Final model-feature registry")
print("-" * 45)
print(f"Pre-redundancy model features: {len(model_feature_columns)}")
print(f"Administrative IDs excluded:  {len(redundant_administrative_ids)}")
print(f"Final model features:          {len(final_model_features)}")
print(f"Quantitative features:         {len(final_quantitative_features)}")
print(f"Categorical features:          {len(final_categorical_features)}")
print(
    f"Administrative IDs retained in df_work: "
    f"{all(column in df_work.columns for column in redundant_administrative_ids)}"
)

assert len(final_model_features) == 44, "Unexpected final model-feature count."

assert len(final_quantitative_features) == 8, "Unexpected quantitative-feature count."

assert len(final_categorical_features) == 36, "Unexpected categorical-feature count."

assert not set(redundant_administrative_ids) & set(
    final_model_features
), "Redundant administrative IDs remain in final model features."

assert set(administrative_description_features).issubset(
    final_model_features
), "Decoded administrative descriptions are missing."

assert set(final_quantitative_features) | set(final_categorical_features) == set(
    final_model_features
), "Final semantic roles do not cover all model features."

assert not set(final_quantitative_features) & set(
    final_categorical_features
), "Final quantitative and categorical feature sets overlap."

assert (
    df_work[final_model_features].isna().sum().sum() == 0
), "Missing values remain in final model features."

print("\nValidation passed: final model-feature registry is clean.")

Final model-feature registry
---------------------------------------------
Pre-redundancy model features: 47
Administrative IDs excluded:  3
Final model features:          44
Quantitative features:         8
Categorical features:          36
Administrative IDs retained in df_work: True

Validation passed: final model-feature registry is clean.


In [26]:
# Construct final pre-encoding modeling frame

modeling_columns = (
    identifier_columns + source_outcome_columns + [target_column] + final_model_features
)

df_model = df_work[modeling_columns].copy()

print("Final pre-encoding modeling frame")
print("-" * 50)
print(f"Rows:                 {df_model.shape[0]:,}")
print(f"Columns:              {df_model.shape[1]}")
print(f"Identifiers:          {len(identifier_columns)}")
print(f"Source outcomes:      {len(source_outcome_columns)}")
print(f"Targets:              1")
print(f"Model features:       {len(final_model_features)}")
print(f"Quantitative:         {len(final_quantitative_features)}")
print(f"Categorical:          {len(final_categorical_features)}")
print(f"Missing cells:        {df_model.isna().sum().sum():,}")
print(f"Duplicate column names: {df_model.columns.duplicated().sum()}")

expected_columns = (
    len(identifier_columns)
    + len(source_outcome_columns)
    + 1
    + len(final_model_features)
)

assert (
    df_model.shape[0] == df_work.shape[0]
), "Modeling-frame row count differs from working data."

assert df_model.shape[1] == expected_columns, "Unexpected modeling-frame column count."

assert (
    not df_model.columns.duplicated().any()
), "Duplicate column names detected in modeling frame."

assert df_model.isna().sum().sum() == 0, "Missing values remain in modeling frame."

assert set(final_model_features).issubset(
    df_model.columns
), "One or more final model features are missing."

assert set(identifier_columns).issubset(
    df_model.columns
), "Identifier columns are missing."

assert set(source_outcome_columns).issubset(
    df_model.columns
), "Source outcome is missing."

assert target_column in df_model.columns, "Target column is missing."

print("\nValidation passed: final pre-encoding modeling frame is complete.")

df_model.head()

Final pre-encoding modeling frame
--------------------------------------------------
Rows:                 101,766
Columns:              48
Identifiers:          2
Source outcomes:      1
Targets:              1
Model features:       44
Quantitative:         8
Categorical:          36
Missing cells:        0
Duplicate column names: 0

Validation passed: final pre-encoding modeling frame is complete.


,encounter_id,patient_nbr,readmitted,readmitted_30d,race,gender,age,time_in_hospital,payer_code,medical_specialty,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,admission_type_description,discharge_disposition_description,admission_source_description
0,2278392,8222157,NO,0,Caucasian,Female,[0-10),1,Unknown,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,Unknown,Not Mapped,Physician Referral
1,149190,55629189,>30,0,Caucasian,Female,[10-20),3,Unknown,Unknown,...,No,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room
2,64410,86047875,NO,0,AfricanAmerican,Female,[20-30),2,Unknown,Unknown,...,No,No,No,No,No,No,Yes,Emergency,Discharged to home,Emergency Room
3,500364,82442376,NO,0,Caucasian,Male,[30-40),2,Unknown,Unknown,...,No,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room
4,16680,42519267,NO,0,Caucasian,Male,[40-50),1,Unknown,Unknown,...,No,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room


In [27]:
# Validate patient grouping structure before train/test partitioning

patient_group_audit = df_model.groupby("patient_nbr").agg(
    encounter_count=("encounter_id", "count"),
    positive_readmissions=("readmitted_30d", "sum"),
)

unique_patients = patient_group_audit.shape[0]
multi_encounter_patients = int((patient_group_audit["encounter_count"] > 1).sum())
max_encounters_per_patient = int(patient_group_audit["encounter_count"].max())

print("Patient-grouping validation")
print("-" * 45)
print(f"Total encounters:              {len(df_model):,}")
print(f"Unique patients:               {unique_patients:,}")
print(f"Patients with >1 encounter:    {multi_encounter_patients:,}")
print(f"Maximum encounters per patient:{max_encounters_per_patient:>6}")

assert unique_patients == 71_518, "Unexpected unique-patient count."

assert (
    multi_encounter_patients == 16_773
), "Repeated-patient count differs from the raw-data audit."

assert (
    max_encounters_per_patient == 40
), "Maximum encounters per patient differs from the audit."

print("\nValidation passed: patient grouping matches the raw-data audit.")

Patient-grouping validation
---------------------------------------------
Total encounters:              101,766
Unique patients:               71,518
Patients with >1 encounter:    16,773
Maximum encounters per patient:    40

Validation passed: patient grouping matches the raw-data audit.


In [28]:
# Create reproducible patient-disjoint train/validation/test partitions

from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# First split: 70% train, 30% temporary holdout
train_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=RANDOM_STATE,
)

train_idx, temp_idx = next(
    train_splitter.split(
        df_model,
        y=df_model[target_column],
        groups=df_model["patient_nbr"],
    )
)

df_train = df_model.iloc[train_idx].copy()
df_temp = df_model.iloc[temp_idx].copy()

# Second split: divide the 30% holdout approximately equally
# into validation and test partitions.
holdout_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=RANDOM_STATE,
)

validation_idx, test_idx = next(
    holdout_splitter.split(
        df_temp,
        y=df_temp[target_column],
        groups=df_temp["patient_nbr"],
    )
)

df_validation = df_temp.iloc[validation_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

train_patients = set(df_train["patient_nbr"])
validation_patients = set(df_validation["patient_nbr"])
test_patients = set(df_test["patient_nbr"])

print("Patient-disjoint dataset partitioning")
print("-" * 55)

for name, frame in [
    ("Train", df_train),
    ("Validation", df_validation),
    ("Test", df_test),
]:
    print(
        f"{name:<12}"
        f"rows={len(frame):>7,} | "
        f"patients={frame['patient_nbr'].nunique():>6,} | "
        f"positive_rate={frame[target_column].mean():.4f}"
    )

print()
print(
    f"Train/validation patient overlap: " f"{len(train_patients & validation_patients)}"
)
print(f"Train/test patient overlap:       " f"{len(train_patients & test_patients)}")
print(
    f"Validation/test patient overlap:  " f"{len(validation_patients & test_patients)}"
)

assert (
    len(train_patients & validation_patients) == 0
), "Patient leakage detected between train and validation."

assert (
    len(train_patients & test_patients) == 0
), "Patient leakage detected between train and test."

assert (
    len(validation_patients & test_patients) == 0
), "Patient leakage detected between validation and test."

assert len(df_train) + len(df_validation) + len(df_test) == len(
    df_model
), "Partition row counts do not reconcile with the modeling frame."

assert train_patients | validation_patients | test_patients == set(
    df_model["patient_nbr"]
), "Patient partitions do not cover the complete modeling population."

print("\nValidation passed: all patients belong to exactly one partition.")

Patient-disjoint dataset partitioning
-------------------------------------------------------
Train       rows= 71,520 | patients=50,062 | positive_rate=0.1122
Validation  rows= 15,027 | patients=10,728 | positive_rate=0.1102
Test        rows= 15,219 | patients=10,728 | positive_rate=0.1104

Train/validation patient overlap: 0
Train/test patient overlap:       0
Validation/test patient overlap:  0

Validation passed: all patients belong to exactly one partition.


### 7D — Final Partition Integrity Audit

Before exporting the processed datasets, perform a final integrity audit of the patient-disjoint train, validation, and test partitions.

This validation confirms that:

- partition row counts reconcile with the complete modeling frame,
- patient assignments remain mutually exclusive,
- all partitions contain both target classes,
- target prevalence remains reasonably stable,
- no missing values were introduced,
- schemas and data types remain consistent,
- encounter identifiers remain unique,
- and every source encounter is represented exactly once.

These checks establish the final preprocessing boundary before persistent modeling artifacts are created.

In [29]:
# Final integrity audit of train/validation/test partitions

partition_frames = {
    "Train": df_train,
    "Validation": df_validation,
    "Test": df_test,
}

reference_columns = list(df_model.columns)
reference_dtypes = df_model.dtypes.astype(str)

audit_records = []

for name, frame in partition_frames.items():
    missing_cells = int(frame.isna().sum().sum())
    duplicate_encounters = int(frame["encounter_id"].duplicated().sum())
    unique_patients_split = int(frame["patient_nbr"].nunique())
    target_classes = sorted(frame[target_column].unique().tolist())
    positive_rate = float(frame[target_column].mean())

    schema_matches = list(frame.columns) == reference_columns
    dtypes_match = frame.dtypes.astype(str).equals(reference_dtypes)

    audit_records.append(
        {
            "partition": name,
            "rows": len(frame),
            "patients": unique_patients_split,
            "positive_rate": positive_rate,
            "target_classes": target_classes,
            "missing_cells": missing_cells,
            "duplicate_encounters": duplicate_encounters,
            "schema_matches": schema_matches,
            "dtypes_match": dtypes_match,
        }
    )

partition_audit = pd.DataFrame(audit_records)

# Reconciliation checks
total_partition_rows = sum(len(frame) for frame in partition_frames.values())

combined_encounter_ids = pd.concat(
    [frame["encounter_id"] for frame in partition_frames.values()],
    ignore_index=True,
)

all_encounters_unique = not combined_encounter_ids.duplicated().any()

encounter_coverage_complete = set(combined_encounter_ids) == set(
    df_model["encounter_id"]
)

patient_overlap = {
    "train_validation": len(train_patients & validation_patients),
    "train_test": len(train_patients & test_patients),
    "validation_test": len(validation_patients & test_patients),
}

overall_positive_rate = df_model[target_column].mean()
max_prevalence_deviation = (
    (partition_audit["positive_rate"] - overall_positive_rate).abs().max()
)

print("Final partition integrity audit")
print("-" * 55)
print(f"Modeling-frame rows:             {len(df_model):,}")
print(f"Partition rows:                  {total_partition_rows:,}")
print(f"Rows reconcile:                  {total_partition_rows == len(df_model)}")
print(f"Encounter coverage complete:     {encounter_coverage_complete}")
print(f"Encounter IDs globally unique:   {all_encounters_unique}")
print(f"Overall positive rate:           {overall_positive_rate:.4f}")
print(f"Maximum prevalence deviation:    {max_prevalence_deviation:.4f}")
print()
print("Patient overlap:")
for pair, count in patient_overlap.items():
    print(f"  {pair:<22} {count:,}")

# Hard validation gates
assert total_partition_rows == len(
    df_model
), "Partition rows do not reconcile with df_model."

assert (
    encounter_coverage_complete
), "Partition encounter IDs do not cover the complete modeling frame."

assert (
    all_encounters_unique
), "At least one encounter appears in more than one partition."

assert all(
    partition_audit["missing_cells"] == 0
), "Missing values were introduced into one or more partitions."

assert all(
    partition_audit["duplicate_encounters"] == 0
), "Duplicate encounter IDs detected within a partition."

assert all(
    partition_audit["schema_matches"]
), "Partition column schemas are inconsistent."

assert all(partition_audit["dtypes_match"]), "Partition data types are inconsistent."

assert all(
    classes == [0, 1] for classes in partition_audit["target_classes"]
), "One or more partitions does not contain both target classes."

assert all(
    count == 0 for count in patient_overlap.values()
), "Patient leakage detected between partitions."

print("\nValidation passed: final partition integrity is clean.")

partition_audit

Final partition integrity audit
-------------------------------------------------------
Modeling-frame rows:             101,766
Partition rows:                  101,766
Rows reconcile:                  True
Encounter coverage complete:     True
Encounter IDs globally unique:   True
Overall positive rate:           0.1116
Maximum prevalence deviation:    0.0014

Patient overlap:
  train_validation       0
  train_test             0
  validation_test        0

Validation passed: final partition integrity is clean.


,partition,rows,patients,positive_rate,target_classes,missing_cells,duplicate_encounters,schema_matches,dtypes_match
0,Train,71520,50062,0.112150,"[0, 1]",0,0,True,True
1,Validation,15027,10728,0.110202,"[0, 1]",0,0,True,True
2,Test,15219,10728,0.110388,"[0, 1]",0,0,True,True


### 7E — Export Processed Modeling Artifacts

Persist the validated patient-disjoint datasets and preprocessing metadata for downstream modeling.

The exported artifacts establish a reproducible boundary between data preparation and model development. Train, validation, and test partitions are stored separately so subsequent modeling workflows do not recreate or alter the validated patient assignments.

Feature-role metadata is also exported to preserve the final modeling schema, including quantitative features, categorical features, identifiers, the source outcome, and the binary 30-day readmission target.

In [30]:
# Export validated processed datasets and feature metadata

from pathlib import Path
import json

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

export_paths = {
    "train": PROCESSED_DIR / "train.csv",
    "validation": PROCESSED_DIR / "validation.csv",
    "test": PROCESSED_DIR / "test.csv",
}

# Persist patient-disjoint partitions
df_train.to_csv(export_paths["train"], index=False)
df_validation.to_csv(export_paths["validation"], index=False)
df_test.to_csv(export_paths["test"], index=False)

feature_registry = {
    "target": target_column,
    "source_outcome": source_outcome_columns,
    "identifiers": identifier_columns,
    "model_features": final_model_features,
    "quantitative_features": final_quantitative_features,
    "categorical_features": final_categorical_features,
}

feature_registry_path = PROCESSED_DIR / "feature_registry.json"

with feature_registry_path.open("w", encoding="utf-8") as file:
    json.dump(feature_registry, file, indent=2)

print("Processed modeling artifacts exported")
print("-" * 50)

for name, path in export_paths.items():
    print(
        f"{name.capitalize():<12} "
        f"{path.name:<20} "
        f"rows={len(partition_frames[name.capitalize()]):,}"
    )

print(f"{'Metadata':<12} {feature_registry_path.name}")
print(f"\nOutput directory: {PROCESSED_DIR}")

Processed modeling artifacts exported
--------------------------------------------------
Train        train.csv            rows=71,520
Validation   validation.csv       rows=15,027
Test         test.csv             rows=15,219
Metadata     feature_registry.json

Output directory: C:\Users\amohe.000\Downloads\healthcare-readmission-analytics\data\processed


### 7F — Reload and Validate Exported Artifacts

Reload the persisted modeling artifacts from disk and validate them independently of the in-memory preprocessing objects.

This final persistence check confirms that serialization preserved:

- partition dimensions and column schemas,
- encounter and patient identifiers,
- the binary 30-day readmission target,
- patient-disjoint partition boundaries,
- feature-registry metadata,
- and complete encounter coverage.

Successful validation establishes the exported files as the authoritative inputs for downstream modeling.

In [31]:
# Reload and independently validate persisted modeling artifacts

with feature_registry_path.open("r", encoding="utf-8") as file:
    reloaded_registry = json.load(file)

reloaded_partitions = {
    "Train": pd.read_csv(export_paths["train"]),
    "Validation": pd.read_csv(export_paths["validation"]),
    "Test": pd.read_csv(export_paths["test"]),
}

expected_rows = {
    "Train": len(df_train),
    "Validation": len(df_validation),
    "Test": len(df_test),
}

print("Persisted-artifact validation")
print("-" * 50)

for name, frame in reloaded_partitions.items():
    print(
        f"{name:<12} "
        f"rows={len(frame):>7,} | "
        f"columns={frame.shape[1]:>2} | "
        f"patients={frame['patient_nbr'].nunique():>6,} | "
        f"positive_rate={frame[target_column].mean():.4f}"
    )

# Validate dimensions and schema
for name, frame in reloaded_partitions.items():
    assert (
        len(frame) == expected_rows[name]
    ), f"{name} row count changed after persistence."

    assert list(frame.columns) == list(
        df_model.columns
    ), f"{name} schema changed after persistence."

    assert (
        frame[target_column].isna().sum() == 0
    ), f"{name} target contains missing values."

    assert set(frame[target_column].unique()) == {
        0,
        1,
    }, f"{name} target is no longer binary."

    assert (
        frame["encounter_id"].duplicated().sum() == 0
    ), f"{name} contains duplicate encounter IDs."

# Validate patient-disjoint boundaries from the persisted files
reloaded_patient_sets = {
    name: set(frame["patient_nbr"]) for name, frame in reloaded_partitions.items()
}

assert not (
    reloaded_patient_sets["Train"] & reloaded_patient_sets["Validation"]
), "Patient overlap detected between persisted train and validation sets."

assert not (
    reloaded_patient_sets["Train"] & reloaded_patient_sets["Test"]
), "Patient overlap detected between persisted train and test sets."

assert not (
    reloaded_patient_sets["Validation"] & reloaded_patient_sets["Test"]
), "Patient overlap detected between persisted validation and test sets."

# Validate complete encounter coverage
reloaded_encounters = pd.concat(
    [frame["encounter_id"] for frame in reloaded_partitions.values()],
    ignore_index=True,
)

assert len(reloaded_encounters) == len(
    df_model
), "Persisted partition rows do not reconcile with df_model."

assert reloaded_encounters.nunique() == len(
    df_model
), "Persisted encounter IDs are not globally unique."

assert set(reloaded_encounters) == set(
    df_model["encounter_id"]
), "Persisted partitions do not contain the complete encounter population."

# Validate feature metadata
assert (
    reloaded_registry == feature_registry
), "Feature registry changed during persistence."

assert set(reloaded_registry["model_features"]) == set(
    final_model_features
), "Persisted model-feature registry is inconsistent."

assert set(reloaded_registry["quantitative_features"]) == set(
    final_quantitative_features
), "Persisted quantitative-feature registry is inconsistent."

assert set(reloaded_registry["categorical_features"]) == set(
    final_categorical_features
), "Persisted categorical-feature registry is inconsistent."

print()
print("Patient overlap:")
print(
    "  train_validation:",
    len(reloaded_patient_sets["Train"] & reloaded_patient_sets["Validation"]),
)
print(
    "  train_test:      ",
    len(reloaded_patient_sets["Train"] & reloaded_patient_sets["Test"]),
)
print(
    "  validation_test: ",
    len(reloaded_patient_sets["Validation"] & reloaded_patient_sets["Test"]),
)

print(f"\nEncounter coverage: {reloaded_encounters.nunique():,}/{len(df_model):,}")
print("Feature registry:   verified")
print("\nValidation passed: persisted modeling artifacts are reproducible.")

Persisted-artifact validation
--------------------------------------------------
Train        rows= 71,520 | columns=48 | patients=50,062 | positive_rate=0.1122
Validation   rows= 15,027 | columns=48 | patients=10,728 | positive_rate=0.1102
Test         rows= 15,219 | columns=48 | patients=10,728 | positive_rate=0.1104

Patient overlap:
  train_validation: 0
  train_test:       0
  validation_test:  0

Encounter coverage: 101,766/101,766
Feature registry:   verified

Validation passed: persisted modeling artifacts are reproducible.


## 7G — Preprocessing Completion Summary

The preprocessing pipeline is complete and has produced validated, reproducible modeling artifacts for 30-day hospital readmission analysis.

### Final dataset

- **101,766 encounters**
- **71,518 unique patients**
- Binary target: `readmitted_30d`
- Positive class: readmission within 30 days
- Overall positive prevalence: **11.16%**

### Data-quality treatment

- Source-defined `?` missing-value markers were standardized.
- `weight` was removed because **96.86%** of observations were missing.
- Missing categorical and diagnosis information was represented explicitly rather than silently imputing unsupported values.
- All **374,017** audited missing cells were reconciled through documented preprocessing decisions.
- Final model-eligible features contain no missing values.

### Administrative features

Administrative ID mappings were parsed from the source metadata and decoded into interpretable descriptions for:

- admission type,
- discharge disposition,
- admission source.

The original ID fields were retained for traceability but excluded from the final modeling feature set after confirming deterministic redundancy with their decoded descriptions.

### Target construction

The original `readmitted` outcome was preserved.

A binary modeling target, `readmitted_30d`, was constructed such that:

- `<30` → `1`
- `NO` or `>30` → `0`

The resulting target contains **11,357 positive encounters (11.16%)**, and row-level validation confirmed exact agreement with the source outcome.

### Modeling feature registry

The final model feature set contains:

- **44 model features**
- **8 quantitative features**
- **36 categorical features**

Identifiers and outcome variables are protected from feature leakage, and audited zero-variance and redundant administrative ID fields were excluded.

### Patient-disjoint partitioning

To prevent repeated-patient leakage, train, validation, and test partitions were created at the patient level.

| Partition | Encounters | Unique patients | Positive rate |
|---|---:|---:|---:|
| Train | 71,520 | 50,062 | 11.22% |
| Validation | 15,027 | 10,728 | 11.02% |
| Test | 15,219 | 10,728 | 11.04% |

Patient overlap between every pair of partitions is **zero**.

### Exported artifacts

The validated modeling artifacts were persisted to `data/processed/`:

- `train.csv`
- `validation.csv`
- `test.csv`
- `feature_registry.json`

The exported files were independently reloaded and validated for schema integrity, encounter coverage, target integrity, patient separation, and feature metadata consistency.

**Notebook 02 status: preprocessing complete and validated.**

The processed artifacts are ready for exploratory analysis and downstream model development.